In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from xgboost import XGBRegressor
import joblib
import os

os.makedirs('../models', exist_ok=True)
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load ML-ready datasets
f1    = pd.read_csv('../data/processed/f1_ml_ready.csv')
cars  = pd.read_csv('../data/processed/cars_ml_ready.csv')
qual  = pd.read_csv('../data/processed/qual_ml_ready.csv')

print("F1 :", f1.shape)
print("Cars:", cars.shape)
print("Qual:", qual.shape)

In [4]:
# ── Cell 2: MPG prediction model (Random Forest) ──

# Features for MPG prediction
features = [
    'engine_displacement_l', 'cylinders', 'displacement_per_cyl',
    'fuel_enc', 'hwy_city_ratio', 'year_rel'
]
target = 'highway_mpg'

# Prepare data
df_cars = cars[features + [target]].dropna()
X = df_cars[features]
y = df_cars[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MPG Predictor — Random Forest")
print(f"  MAE : {mae:.2f} MPG")
print(f"  R²  : {r2:.3f}")

# Feature importance
fig, ax = plt.subplots(figsize=(8, 4))
importance = pd.Series(rf.feature_importances_, index=features).sort_values()
importance.plot(kind='barh', color='#e10600', ax=ax)
ax.set_title('Feature importance — MPG predictor')
plt.tight_layout()
plt.savefig('../data/processed/plot_mpg_importance.png', dpi=150)
plt.show()

# Save model
joblib.dump(rf, '../models/mpg_predictor.pkl')
print("Model saved to ../models/mpg_predictor.pkl")

In [5]:
# ── Cell 3: Lap time predictor (XGBoost) ──

features = [
    'Compound_enc', 'TyreLife', 'SpeedST_norm',
    'S1_ratio', 'S2_ratio', 'S3_ratio',
    'LapNumber', 'Year_rel', 'Team_enc'
]
target = 'LapTime_delta'

df_f1 = f1[features + [target]].dropna()
X = df_f1[features]
y = df_f1[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
xgb = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train)

# Evaluate
y_pred = xgb.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"Lap Time Predictor — XGBoost")
print(f"  MAE : {mae:.2f} seconds")
print(f"  R²  : {r2:.3f}")

# Feature importance
fig, ax = plt.subplots(figsize=(8, 4))
importance = pd.Series(xgb.feature_importances_, index=features).sort_values()
importance.plot(kind='barh', color='#e10600', ax=ax)
ax.set_title('Feature importance — lap time predictor')
plt.tight_layout()
plt.savefig('../data/processed/plot_laptime_importance.png', dpi=150)
plt.show()

joblib.dump(xgb, '../models/laptime_predictor.pkl')
print("Model saved to ../models/laptime_predictor.pkl")

In [6]:
# ── Cell 4: Win probability model ──
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create win label — P1 in qualifying = potential winner
# Use top 3 finish as proxy (we don't have race results, using qual position)
q_model = qual.copy()
q_model = q_model.dropna(subset=['BestQual', 'GapToPole', 'QualPos', 'RecentForm'])

# Target: qualified in top 3 (podium contender)
q_model['PodiumContender'] = (q_model['QualPos'] <= 3).astype(int)

features = ['BestQual', 'GapToPole', 'QualPos', 'TeamAvgQual', 'RecentForm']
target   = 'PodiumContender'

X = q_model[features]
y = q_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
gbc = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                  learning_rate=0.05, random_state=42)
gbc.fit(X_train, y_train)

# Evaluate
y_pred = gbc.predict(X_test)
y_prob = gbc.predict_proba(X_test)[:, 1]
acc    = accuracy_score(y_test, y_pred)

print(f"Win Probability Model — Gradient Boosting")
print(f"  Accuracy: {acc:.3f}")
print(f"\nClassification report:")
print(classification_report(y_test, y_pred))

# Show probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_prob[y_test == 0], bins=30, alpha=0.7,
             color='#1e1e2e', label='Not podium', edgecolor='none')
axes[0].hist(y_prob[y_test == 1], bins=30, alpha=0.7,
             color='#e10600', label='Podium', edgecolor='none')
axes[0].set_title('Predicted podium probability distribution')
axes[0].set_xlabel('Probability')
axes[0].legend()

# Feature importance
importance = pd.Series(gbc.feature_importances_, index=features).sort_values()
importance.plot(kind='barh', color='#e10600', ax=axes[1])
axes[1].set_title('Feature importance — win probability')

plt.tight_layout()
plt.savefig('../data/processed/plot_winprob.png', dpi=150)
plt.show()

joblib.dump(gbc, '../models/win_probability.pkl')
print("Model saved to ../models/win_probability.pkl")

In [7]:
# ── Cell 4 (fixed): Win probability model ──
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

q_model = qual.copy()
q_model = q_model.dropna(subset=['BestQual', 'GapToPole', 'RecentForm', 'TeamAvgQual'])
q_model['PodiumContender'] = (q_model['QualPos'] <= 3).astype(int)

# Removed QualPos — it directly encodes the target
features = ['BestQual', 'GapToPole', 'TeamAvgQual', 'RecentForm']
target   = 'PodiumContender'

X = q_model[features]
y = q_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

gbc = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                  learning_rate=0.05, random_state=42)
gbc.fit(X_train, y_train)

y_pred = gbc.predict(X_test)
y_prob = gbc.predict_proba(X_test)[:, 1]
acc    = accuracy_score(y_test, y_pred)

print(f"Win Probability Model — Gradient Boosting")
print(f"  Accuracy: {acc:.3f}")
print(f"\nClassification report:")
print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y_prob[y_test == 0], bins=30, alpha=0.7,
             color='#1e1e2e', label='Not podium', edgecolor='none')
axes[0].hist(y_prob[y_test == 1], bins=30, alpha=0.7,
             color='#e10600', label='Podium', edgecolor='none')
axes[0].set_title('Predicted podium probability distribution')
axes[0].set_xlabel('Probability')
axes[0].legend()

importance = pd.Series(gbc.feature_importances_, index=features).sort_values()
importance.plot(kind='barh', color='#e10600', ax=axes[1])
axes[1].set_title('Feature importance — win probability')

plt.tight_layout()
plt.savefig('../data/processed/plot_winprob.png', dpi=150)
plt.show()

joblib.dump(gbc, '../models/win_probability.pkl')
print("Model saved to ../models/win_probability.pkl")